# Phase 2.1 — Scope-aware SAP Retrieval
Study-level and target-level retrieval; original spans and exclusions audited. Runs Phase 2 baseline for comparison. No API key; confidential source text remains local.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'generic-retrieval-v2'
ROOT = Path('/content/tfl-generic-retrieval-v2')
if not ROOT.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)])
else:
    current = subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'], text=True).strip()
    assert current == BRANCH, f'Wrong branch: {current}'
    subprocess.check_call(['git','-C',str(ROOT),'pull','--ff-only'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[notebook,test]','sentence-transformers>=3,<6'])
SRC = (ROOT / 'src').resolve()
assert (SRC / 'tfl_agent/tools/sap_hybrid.py').is_file(), SRC
if str(SRC) in sys.path:
    sys.path.remove(str(SRC))
sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
cached = sys.modules.get('tfl_agent')
if cached is not None and not Path(cached.__file__).resolve().is_relative_to(SRC):
    raise RuntimeError('Wrong tfl_agent checkout already imported. Restart session and run all.')
import tfl_agent
from tfl_agent.agent.sap_scope_pilot import run_scope_retrieval
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC)
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent package loaded:', tfl_agent.__file__)

## 1. Configure the existing Drive SAP

For SAP_001_00.pdf use physical pages 4–21, excluding the Content List and shells. For a different study, edit the range and targets together. These targets are the previous demo targets; the failed Phase 1 in-library cases are not assumed to belong to this SAP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAP_RELATIVE_PATH = 'Colab Notebooks/agent/midterm demo/SAP_001_00.pdf'
SAP_PATH = Path('/content/drive/MyDrive') / DRIVE_SAP_RELATIVE_PATH
SAP_BODY_PAGE_RANGE = (4, 21)
TARGETS = [
    {'tfl_id':'14.1.3.1','title':'Demographic and Baseline Characteristics – Safety Analysis Set'},
    {'tfl_id':'14.2.3.1','title':'Summary of Progression Free Survival – ITT Analysis Set'},
]
if not SAP_PATH.is_file():
    available = sorted(p.name for p in SAP_PATH.parent.glob('*.pdf'))
    raise FileNotFoundError(f'{SAP_PATH}; PDFs in folder: {available}')
SAVE_TO_DRIVE = True
print('SAP:', SAP_PATH)
print('Physical pages:', SAP_BODY_PAGE_RANGE)

## 2. Run scope-aware retrieval and Phase 2 comparison
The same local embedding model is reused. Study evidence is retrieved once; target evidence and unclassified context remain separate.

In [ ]:
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = Path('/content/SAP_Scope_Phase21') / RUN_ID
summary, rows = run_scope_retrieval(SAP_PATH, SAP_BODY_PAGE_RANGE, TARGETS, OUT)
print(json.dumps(summary, indent=2))
df = pd.DataFrame(rows)
display(df[(df.target_id == 'STUDY') & (df.method == 'hybrid') & (df['rank'] == 1)][['facet','pool','pages','section','text']])
for t in TARGETS:
    display(df[(df.target_id == t['tfl_id']) & (df.method == 'hybrid') & (df['rank'] == 1)][['facet','pool','pages','section','topic_scope','text']])


## 3. Compare scope and provenance
Counts are diagnostics, not relevance accuracy. Baseline and scope-aware retrieval use the same SAP, model and ranking parameters.

In [ ]:
baseline = json.loads((OUT/'Phase2_Baseline/Summary.json').read_text())
display(pd.DataFrame([{'version':'Phase 2','chunks':baseline['chunk_count'],'sections':baseline['section_count'],'unsectioned':baseline['unsectioned_chunks']},{'version':'Phase 2.1','chunks':summary['chunks'],'sections':summary['sections_detected'],'unsectioned':summary['unsectioned_chunks']}]))
chunks = json.loads((OUT/'Scoped_Chunks.json').read_text())
blocks = {b['block_id']:b for b in json.loads((OUT/'SAP_Blocks.json').read_text())}
for c in chunks:
    reconstructed = '\n'.join(blocks[s['block_id']]['text'][s['start']:s['end']] for s in c['source_spans'])
    assert reconstructed == c['text'], c['chunk_id']
print('All original source spans verified.')
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_sap_scope_phase21.py','tests/test_sap_hybrid_phase2.py','tests/test_generic_library_phase1.py'],cwd=ROOT)


## 4. Export results
Save timestamped ZIP in the original private Drive folder under Phase21_Outputs, and download.

In [ ]:
import shutil
from google.colab import files
ZIP_PATH = Path(shutil.make_archive(str(OUT), 'zip', OUT))
print('Local output:', OUT)
print('ZIP:', ZIP_PATH)
if SAVE_TO_DRIVE:
    destination = SAP_PATH.parent / 'Phase21_Outputs'
    destination.mkdir(parents=True, exist_ok=True)
    saved = Path(shutil.copy2(ZIP_PATH, destination / ZIP_PATH.name))
    print('Saved to Google Drive:', saved)
files.download(str(ZIP_PATH))